In [1]:
!pip install psycopg2-binary

  Using cached psycopg2-binary-2.9.10.tar.gz (385 kB)
  Preparing metadata (setup.py) ... done
  DEPRECATION: Building 'psycopg2-binary' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'psycopg2-binary'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for psycopg2-binary: filename=psycopg2_binary-2.9.10-cp310-cp310-macosx_11_0_arm64.whl size=133604 sha256=09cc5e9fc52d2b7c57639007e6e3f224ad83541a9b4f2db4cffc0407f30c8e47
  Stored in directory: /Users/nagurshareefshaik/Library/Caches/pip/wheels/bd/8b/58/ff5a880b960d71f645b3679b454f33612254cc39ecdcd6cc5a
Successfully built psycopg2-binary


In [2]:
import psycopg2
from psycopg2 import sql

# Database connection parameters
DB_CONFIG = {
    "dbname": "healthcare",
    "user": "nagurshareefshaik",
    "password": "",   # Add password if required, otherwise leave empty
    "host": "localhost",
    "port": 5432
}

def connect_db():
    """Establish a connection to PostgreSQL."""
    try:
        conn = psycopg2.connect(**DB_CONFIG)
        conn.autocommit = True
        print("✅ Connected to the database.")
        return conn
    except Exception as e:
        print("❌ Connection failed:", e)
        return None

def create_table(conn):
    """Create the patients table if it does not exist."""
    query = """
    CREATE TABLE IF NOT EXISTS patients (
        patient_id SERIAL PRIMARY KEY,
        name TEXT NOT NULL,
        age INT CHECK (age >= 0),
        gender VARCHAR(10) CHECK (gender IN ('Male','Female')),
        blood_type VARCHAR(3),
        medical_condition TEXT,
        date_of_admission DATE NOT NULL,
        doctor TEXT,
        hospital TEXT,
        insurance_provider TEXT,
        billing_amount NUMERIC(10,2),
        room_number VARCHAR(10),
        admission_type VARCHAR(15) CHECK (admission_type IN ('Emergency','Elective','Urgent')),
        discharge_date DATE,
        medication TEXT,
        test_results VARCHAR(20) CHECK (test_results IN ('Normal','Abnormal','Inconclusive'))
    );
    """
    with conn.cursor() as cur:
        cur.execute(query)
        print("✅ Table created (if not exists).")

import csv
import psycopg2

def insert_from_csv(conn, csv_file_path):
    """
    Insert multiple patient records from a CSV file.
    
    Args:
        conn: psycopg2 connection object
        csv_file_path: Path to the CSV file
    """
    query = """
    INSERT INTO patients (
        name, age, gender, blood_type, medical_condition, date_of_admission,
        doctor, hospital, insurance_provider, billing_amount, room_number,
        admission_type, discharge_date, medication, test_results
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s);
    """
    
    with open(csv_file_path, newline='', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile)
        with conn.cursor() as cur:
            for row in reader:
                data = (
                    row['Name'],
                    int(row['Age']) if row['Age'] else None,
                    row['Gender'],
                    row['Blood Type'],
                    row['Medical Condition'],
                    row['Date of Admission'],
                    row['Doctor'],
                    row['Hospital'],
                    row['Insurance Provider'],
                    float(row['Billing Amount']) if row['Billing Amount'] else None,
                    row['Room Number'],
                    row['Admission Type'],
                    row['Discharge Date'] if row['Discharge Date'] else None,
                    row['Medication'],
                    row['Test Results']
                )
                cur.execute(query, data)
            conn.commit()
            print(f"✅ Data from '{csv_file_path}' inserted successfully.")


def fetch_data(conn):
    """Fetch and display patient records."""
    query = "SELECT patient_id, name, age, medical_condition, billing_amount FROM patients LIMIT 10;"
    with conn.cursor() as cur:
        cur.execute(query)
        rows = cur.fetchall()
        print("📋 Patient Records:")
        for row in rows:
            print(row)

if __name__ == "__main__":
    conn = connect_db()
    if conn:
        create_table(conn)
        insert_from_csv(conn, "data/raw/healthcare/healthcare_dataset.csv")
        fetch_data(conn)
        conn.close()


✅ Connected to the database.
✅ Table created (if not exists).
✅ Data from 'data/raw/healthcare/healthcare_dataset.csv' inserted successfully.
📋 Patient Records:
(1, 'Bobby JacksOn', 30, 'Cancer', Decimal('18856.28'))
(2, 'LesLie TErRy', 62, 'Obesity', Decimal('33643.33'))
(3, 'DaNnY sMitH', 76, 'Obesity', Decimal('27955.10'))
(4, 'andrEw waTtS', 28, 'Diabetes', Decimal('37909.78'))
(5, 'adrIENNE bEll', 43, 'Cancer', Decimal('14238.32'))
(6, 'EMILY JOHNSOn', 36, 'Asthma', Decimal('48145.11'))
(7, 'edwArD EDWaRDs', 21, 'Diabetes', Decimal('19580.87'))
(8, 'CHrisTInA MARtinez', 20, 'Cancer', Decimal('45820.46'))
(9, 'JASmINe aGuIlaR', 82, 'Asthma', Decimal('50119.22'))
(10, 'ChRISTopher BerG', 58, 'Cancer', Decimal('19784.63'))


In [3]:
!pip install pandas sqlalchemy mysql-connector-python

In [14]:
import mysql.connector
from mysql.connector import errorcode
import pandas as pd
import os

# --- Configuration ---
# ⚠️ IMPORTANT: Update these with your MySQL credentials.
DB_CONFIG = {
    "user": "root",         # <-- Change to your MySQL username (e.g., 'root')
    "password": "ghs@195L", # <-- Change to your MySQL password
    "host": "localhost"
}
DB_NAME = "walmart"
TABLE_NAME = "sales"
CSV_FILE_PATH = os.path.join("data/raw/retail", "walmart.csv")

def connect_and_setup_db():
    """
    Connects to MySQL, creates the database if it doesn't exist,
    and returns a connection to the specific database.
    """
    try:
        # Connect to the server first
        conn = mysql.connector.connect(**DB_CONFIG)
        cursor = conn.cursor()
        print("✅ Connected to MySQL server.")
        
        # Create the database if it doesn't exist
        cursor.execute(f"CREATE DATABASE IF NOT EXISTS {DB_NAME} DEFAULT CHARACTER SET 'utf8'")
        print(f"✅ Database '{DB_NAME}' is ready.")
        
        # Connect to the specific database
        conn.database = DB_NAME
        return conn

    except mysql.connector.Error as err:
        if err.errno == errorcode.ER_ACCESS_DENIED_ERROR:
            print("❌ Access denied. Check your username or password in DB_CONFIG.")
        else:
            print(f"❌ Database connection failed: {err}")
        return None

def create_sales_table(conn):
    """Creates the walmart_sales table with the correct schema."""
    query = f"""
    CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
        Store INT,
        Date DATE,
        Weekly_Sales DECIMAL(12, 2),
        Holiday_Flag BOOLEAN,
        Temperature DECIMAL(5, 2),
        Fuel_Price DECIMAL(5, 3),
        CPI DECIMAL(10, 7),
        Unemployment DECIMAL(5, 3),
        PRIMARY KEY (Store, Date)
    );
    """
    try:
        cursor = conn.cursor()
        cursor.execute(query)
        print(f"✅ Table '{TABLE_NAME}' created successfully (if not exists).")
    except mysql.connector.Error as err:
        print(f"❌ Failed to create table: {err}")

def load_data_from_csv(conn, file_path, chunk_size=5000):
    """
    Reads the Walmart CSV in chunks, transforms data types,
    and loads it into the database.
    """
    print(f"⏳ Starting to load data from '{file_path}'...")
    try:
        total_rows_inserted = 0
        # Read CSV in chunks to handle large files and avoid memory issues
        for chunk in pd.read_csv(file_path, chunksize=chunk_size):
            # 1. Convert the 'Date' column from dd-mm-yyyy to yyyy-mm-dd format
            chunk['Date'] = pd.to_datetime(chunk['Date'], format='%d-%m-%Y').dt.date

            # 2. Prepare data for insertion (list of tuples)
            data_to_insert = [tuple(row) for row in chunk.itertuples(index=False)]
            
            # 3. Create the insert query
            cols = ", ".join([f"`{c}`" for c in chunk.columns])
            placeholders = ", ".join(["%s"] * len(chunk.columns))
            query = f"INSERT INTO {TABLE_NAME} ({cols}) VALUES ({placeholders})"

            # 4. Execute the insert command
            cursor = conn.cursor()
            cursor.executemany(query, data_to_insert)
            conn.commit()

            total_rows_inserted += cursor.rowcount
            print(f"  - Inserted a chunk. Total rows so far: {total_rows_inserted}")
        
        print(f"✅ Successfully inserted a total of {total_rows_inserted} rows.")
    except Exception as e:
        print(f"❌ An error occurred during data loading: {e}")
        conn.rollback()

def verify_data(conn):
    """Fetches and displays a few sample records to verify the load."""
    print("\n📋 Verifying loaded data with a sample...")
    try:
        cursor = conn.cursor(dictionary=True) # Fetch as dictionaries for nice printing
        cursor.execute(f"SELECT * FROM {TABLE_NAME} ORDER BY Store, Date LIMIT 5;")
        rows = cursor.fetchall()
        
        if not rows:
            print("  - No data found in the table.")
            return

        print("--- First 5 Records ---")
        for row in rows:
            print(row)
            
    except Exception as e:
        print(f"❌ Could not fetch verification data: {e}")


# --- Main Execution Block ---
if __name__ == "__main__":
    conn = connect_and_setup_db()
    if conn:
        create_sales_table(conn)
        load_data_from_csv(conn, CSV_FILE_PATH)
        verify_data(conn)
        
        conn.close()
        print("\n🎉 Process complete. Database connection closed.")

✅ Connected to MySQL server.
✅ Database 'walmart' is ready.
✅ Table 'sales' created successfully (if not exists).
⏳ Starting to load data from 'data/raw/retail/walmart.csv'...
  - Inserted a chunk. Total rows so far: 5000
  - Inserted a chunk. Total rows so far: 6435
✅ Successfully inserted a total of 6435 rows.

📋 Verifying loaded data with a sample...
--- First 5 Records ---
{'Store': 1, 'Date': datetime.date(2010, 2, 5), 'Weekly_Sales': Decimal('1643690.90'), 'Holiday_Flag': 0, 'Temperature': Decimal('42.31'), 'Fuel_Price': Decimal('2.572'), 'CPI': Decimal('211.0963582'), 'Unemployment': Decimal('8.106')}
{'Store': 1, 'Date': datetime.date(2010, 2, 12), 'Weekly_Sales': Decimal('1641957.44'), 'Holiday_Flag': 1, 'Temperature': Decimal('38.51'), 'Fuel_Price': Decimal('2.548'), 'CPI': Decimal('211.2421698'), 'Unemployment': Decimal('8.106')}
{'Store': 1, 'Date': datetime.date(2010, 2, 19), 'Weekly_Sales': Decimal('1611968.17'), 'Holiday_Flag': 0, 'Temperature': Decimal('39.93'), 'Fuel_P

In [26]:
# db_handler_service.py

import sqlalchemy
from sqlalchemy import create_engine, inspect, text
import pandas as pd


class DBHandlerService:
    def __init__(self, db_type: str, username: str = None, password: str = None,
                 host: str = None, port: int = None, database: str = None, sqlite_path: str = None):
        """
        db_type: "mysql", "postgresql", or "sqlite"
        """
        if db_type == "mysql":
            self.connection_url = f"mysql+mysqlconnector://{username}:{password}@{host}:{port}/{database}"
        elif db_type == "postgresql":
            self.connection_url = f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
        elif db_type == "sqlite":
            self.connection_url = f"sqlite:///{sqlite_path}"
        else:
            raise ValueError("Unsupported DB type. Use mysql, postgresql, or sqlite.")

        self.engine = create_engine(self.connection_url)
        self.conn = self.engine.connect()
        self.inspector = inspect(self.engine)

    def execute_query(self, query: str):
        """Run arbitrary SQL and return as DataFrame"""
        result = self.conn.execute(text(query))
        return pd.DataFrame(result.fetchall(), columns=result.keys())

    def insert(self, table: str, data: dict):
        """Insert a record"""
        keys = ", ".join(data.keys())
        values = ", ".join([f":{k}" for k in data.keys()])
        query = text(f"INSERT INTO {table} ({keys}) VALUES ({values})")
        self.conn.execute(query, data)
        self.conn.commit()

    def update(self, table: str, data: dict, condition: str):
        """Update a record with condition"""
        set_clause = ", ".join([f"{k}=:{k}" for k in data.keys()])
        query = text(f"UPDATE {table} SET {set_clause} WHERE {condition}")
        self.conn.execute(query, data)
        self.conn.commit()

    def delete(self, table: str, condition: str):
        """Delete with condition"""
        query = text(f"DELETE FROM {table} WHERE {condition}")
        self.conn.execute(query)
        self.conn.commit()

    def get_schema(self):
        """Get schema info (tables, columns, datatypes)"""
        schema = {}
        for table in self.inspector.get_table_names():
            columns = self.inspector.get_columns(table)
            schema[table] = {col["name"]: str(col["type"]) for col in columns}
        return schema

    def get_data_dictionary(self, sample_domain_limit: int = 10):
        """
        Build a rich data dictionary:
        - Tables
        - Columns
        - Data Types
        - Primary Keys, Foreign Keys
        - Row Count
        - Distinct Values (domains) [sampled]
        - Min/Max/Null Count for numeric/date columns
        - Relationships (FKs)
        """
        datadict = {}

        for table in self.inspector.get_table_names():
            datadict[table] = {}

            # --- Columns & types
            columns = self.inspector.get_columns(table)
            datadict[table]["columns"] = {
                col["name"]: {
                    "type": str(col["type"]),
                    "nullable": col.get("nullable", True),
                    "default": col.get("default", None)
                } for col in columns
            }

            # --- Primary keys
            pk = self.inspector.get_pk_constraint(table).get("constrained_columns", [])
            datadict[table]["primary_key"] = pk

            # --- Foreign keys
            fks = self.inspector.get_foreign_keys(table)
            datadict[table]["foreign_keys"] = [
                {
                    "column": fk["constrained_columns"],
                    "referred_table": fk["referred_table"],
                    "referred_columns": fk["referred_columns"]
                }
                for fk in fks
            ]

            # --- Row count
            count_query = f"SELECT COUNT(*) as cnt FROM {table}"
            cnt = self.conn.execute(text(count_query)).fetchone()[0]
            datadict[table]["row_count"] = cnt

            # --- Column stats
            col_stats = {}
            for col in datadict[table]["columns"].keys():
                stats = {}
                # unique values count
                q = f"SELECT COUNT(DISTINCT {col}) FROM {table}"
                stats["unique_values_count"] = self.conn.execute(text(q)).fetchone()[0]

                # sample distinct values (domain)
                q = f"SELECT DISTINCT {col} FROM {table} LIMIT {sample_domain_limit}"
                try:
                    domain_vals = [row[0] for row in self.conn.execute(text(q)).fetchall()]
                    stats["sample_values"] = domain_vals
                except Exception:
                    stats["sample_values"] = []

                # min/max/null count (only for numeric/date/time)
                try:
                    q = f"SELECT MIN({col}), MAX({col}), COUNT(*) - COUNT({col}) as null_count FROM {table}"
                    min_val, max_val, nulls = self.conn.execute(text(q)).fetchone()
                    stats["min"] = min_val
                    stats["max"] = max_val
                    stats["null_count"] = nulls
                except Exception:
                    pass

                col_stats[col] = stats

            datadict[table]["column_stats"] = col_stats

        return datadict


    def close(self):
        self.conn.close()
        self.engine.dispose()


In [28]:
# PostgreSQL example
service = DBHandlerService(
    db_type="mysql",
    username="root",
    password="ghs@195L",
    host="localhost",
    port=5432,
    database="walmart"
)

# Run query
print(service.execute_query("SELECT * FROM patients LIMIT 5"))

# Insert
# service.insert("users", {"id": 1, "name": "John", "email": "john@example.com"})

# Schema
print(service.get_schema())

# Data dictionary
print(service.get_data_dictionary())

service.close()


InterfaceError: (mysql.connector.errors.InterfaceError) 2003: Can't connect to MySQL server on '195L@localhost:5432' (Errno 8: nodename nor servname provided, or not known)
(Background on this error at: https://sqlalche.me/e/20/rvf5)

In [1]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
import joblib
import json
import matplotlib.pyplot as plt
import seaborn as sns

# 3.1. Data Loading
print("Loading data...")
data = pd.read_csv('/Users/nagurshareefshaik/Desktop/PhD-CS-GSU/UST_D3CODE-PRISM/PRISM/data/processed/cdd62815-a04f-40a0-984a-44064ab2c558_modeling_dataset.csv')

# 3.2. Preprocessing
print("Preprocessing data...")
# Handle missing values
for column in data.select_dtypes(include=['number']):
    data[column].fillna(data[column].median(), inplace=True)
for column in data.select_dtypes(include=['object']):
    data[column].fillna(data[column].mode()[0], inplace=True)

# One-hot encode categorical variables
data = pd.get_dummies(data, drop_first=True)

# Standardize numerical features
scaler = StandardScaler()
numerical_features = data.drop(columns=['status']).select_dtypes(include=['number']).columns
data[numerical_features] = scaler.fit_transform(data[numerical_features])

# 3.3. Feature Selection
selected_features = ['amount', 'duration', 'payments'] + [f'A{i}' for i in range(2, 17)]
X = data[selected_features]
y = data['status']

# 3.4. Model Training
print("Splitting data into training and testing sets...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training the Logistic Regression model...")
model = LogisticRegression()
model.fit(X_train, y_train)

# 3.5. Evaluation
print("Evaluating the model...")
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Generate confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)

# Print evaluation metrics
print("Evaluation Metrics:")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1}")

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.savefig('artifacts/confusion_matrix.png')
plt.close()

# 3.6. Artifact Saving
if not os.path.exists('artifacts'):
    os.makedirs('artifacts')

print("Saving model and evaluation metrics...")
joblib.dump(model, 'artifacts/loan_approval_model.pkl')

metrics = {
    'accuracy': accuracy,
    'precision': precision,
    'recall': recall,
    'f1_score': f1,
    'confusion_matrix': conf_matrix.tolist()
}
with open('artifacts/evaluation_metrics.json', 'w') as f:
    json.dump(metrics, f)

print("All tasks completed successfully.")

Loading data...
Preprocessing data...


/var/folders/cv/flgh8s7960bc7q380pyn0c6m0000gn/T/ipykernel_92335/1132242859.py:20: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data[column].fillna(data[column].median(), inplace=True)
/var/folders/cv/flgh8s7960bc7q380pyn0c6m0000gn/T/ipykernel_92335/1132242859.py:20: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting v

KeyError: "['status'] not found in axis"